# Chapter 6 -- 第一个分类器：朴素贝叶斯（Naive Bayes）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaosfrey-arch/news-classification-tutorial/blob/main/chapter06_naive_bayes.ipynb)

**本章目标：** 训练第一个真实的分类器，并学会读懂评估结果。

**预计时间：** 60 分钟

> 参考：[sklearn MultinomialNB 文档](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html) | [朴素贝叶斯直觉讲解（StatQuest）](https://www.youtube.com/watch?v=O2L2Uv9pdDA) | [混淆矩阵与 F1-Score 解读](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-report)

> 参考：[sklearn MultinomialNB 文档](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html) | [朴素贝叶斯直觉讲解（StatQuest YouTube）](https://www.youtube.com/watch?v=O2L2Uv9pdDA) | [混淆矩阵与 F1-Score 解读](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-report)

## 6.1 从概率说起

**概率** 是衡量某件事「可能发生」的程度，范围 0 到 1（或 0% 到 100%）。

- P(明天下雨) = 0.7 → 70% 的可能性下雨
- P(掷骰子得到 6) = 1/6 ≈ 0.17

朴素贝叶斯用概率来分类：**看到这篇文章里的词，每个类别的概率是多少？**

## 6.2 贝叶斯定理

核心问题：看到一篇文章里有词 "quarterback"，它属于 Sports 类的概率是多少？

$$P(Sports \mid \text{包含 quarterback}) = \frac{P(\text{包含 quarterback} \mid Sports) \times P(Sports)}{P(\text{包含 quarterback})}$$

用人话说：
- **P(Sports)** = 先验概率：不看文章内容，本来有多少比例是体育新闻？（约 25%）
- **P(包含 quarterback | Sports)** = 体育新闻里出现 "quarterback" 的频率高不高？（高！）
- **结果** = 综合考虑后，这篇文章是体育的概率

对所有 4 个类别都这样算，概率最高的类别就是预测结果。

## 6.3 为什么叫「朴素」（Naive）？

假设文章里所有词**相互独立**，互不影响。

现实中这当然不对——"artificial" 和 "intelligence" 经常一起出现。但即使这个假设是错的，模型在实践中效果依然不错。

这就是「朴素」的含义：用了一个简单但不完全正确的假设，换来了极快的计算速度。

## 6.4 准备数据（复用 Chapter 5 的代码）

In [ ]:
import pandas as pd, numpy as np, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'http\S+|www\S+|https\S+', '', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df = pd.read_csv('dataset.csv').dropna(subset=['Class']).copy()
df['Description'] = df['Description'].fillna('')
df['text'] = (df['Title'] + ' ' + df['Description']).apply(clean_text)
le = LabelEncoder()
y = le.fit_transform(df['Class'])
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values, y, test_size=0.3, random_state=42, stratify=y)

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True, min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
print('数据准备完毕！')

## 6.5 训练朴素贝叶斯

In [ ]:
from sklearn.naive_bayes import MultinomialNB

# alpha=0.1 是平滑参数（Laplace Smoothing）
# 作用：防止「如果训练时没见过这个词，概率就是 0」的问题
# alpha 越大，平滑越强；alpha=0.1 比默认值 1.0 更精准
nb_model = MultinomialNB(alpha=0.1)

# 训练（fit）
nb_model.fit(X_train_tfidf, y_train)

print('朴素贝叶斯训练完成！')
print(f'模型学到了 {X_train_tfidf.shape[1]:,} 个特征（词）的统计规律')

In [ ]:
# 预测
y_pred_nb = nb_model.predict(X_test_tfidf)

# 看几个具体预测
print('前 5 个测试样本的预测结果：')
for i in range(5):
    true_label = le.classes_[y_test[i]]
    pred_label = le.classes_[y_pred_nb[i]]
    correct = 'O' if true_label == pred_label else 'X'
    print(f'  [{correct}] 真实：{true_label:15s} 预测：{pred_label}')

## 6.6 评估指标详解

### Accuracy（准确率）

**定义：** 预测正确的比例

$$\text{Accuracy} = \frac{\text{预测正确的数量}}{\text{总数量}}$$

**局限：** 当类别不平衡时，Accuracy 会误导。（本数据集平衡，所以 Accuracy 可靠。）

### Precision（精确率）和 Recall（召回率）

以「Sports」类为例：

**Precision（精确率）：**
> 我预测为 Sports 的文章里，真的是 Sports 的有多少？

$$\text{Precision} = \frac{\text{真 Sports 且预测 Sports}}{\text{所有预测为 Sports 的}}$$

**Recall（召回率）：**
> 所有真正的 Sports 文章里，我找到了多少？

$$\text{Recall} = \frac{\text{真 Sports 且预测 Sports}}{\text{所有真正是 Sports 的}}$$

**F1-Score：** Precision 和 Recall 的调和平均，两者都高 F1 才高。
$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# 基本指标
acc_nb  = accuracy_score(y_test, y_pred_nb)
f1_nb   = f1_score(y_test, y_pred_nb, average='weighted')
prec_nb = precision_score(y_test, y_pred_nb, average='weighted')
rec_nb  = recall_score(y_test, y_pred_nb, average='weighted')

print('=== Naive Bayes 测试集结果 ===')
print(f'  Accuracy          : {acc_nb:.4f} ({acc_nb:.1%})')
print(f'  Weighted F1-Score : {f1_nb:.4f}')
print(f'  Weighted Precision: {prec_nb:.4f}')
print(f'  Weighted Recall   : {rec_nb:.4f}')
print('\n=== 逐类别详细报告 ===')
print(classification_report(y_test, y_pred_nb, target_names=le.classes_))

In [ ]:
# 混淆矩阵：行=真实类别，列=预测类别，对角线=预测正确
cm = confusion_matrix(y_test, y_pred_nb)

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Naive Bayes -- Confusion Matrix')
plt.tight_layout()
plt.show()

# 分析最大的错误
print('最多的误判：')
np.fill_diagonal(cm, 0)  # 把对角线（正确预测）清零
max_err = np.unravel_index(cm.argmax(), cm.shape)
print(f'  把 {le.classes_[max_err[0]]} 误判为 {le.classes_[max_err[1]]}: {cm[max_err]} 次')
print('原因：这两类在词汇上有一定重叠（科技公司新闻可能同时涉及商业和科技）')

## 6.7 保存模型

In [ ]:
import joblib

# joblib 专门用来保存 sklearn 模型
joblib.dump(nb_model, 'naive_bayes_model.joblib')
joblib.dump(tfidf, 'tfidf_vectorizer.joblib')
joblib.dump(le, 'label_encoder.joblib')
print('模型已保存！')

## 练习

In [ ]:
# 练习：用训练好的模型预测下面这段文字属于哪个类别
test_article = "The Olympic champion won three gold medals in swimming at the World Championships"

# 步骤：
# 1. clean_text(test_article)
# 2. tfidf.transform([cleaned])
# 3. nb_model.predict(transformed)
# 4. le.inverse_transform(prediction)

# 你的代码：


## 总结

**Naive Bayes 结果：Accuracy ≈ 90%**

这对一个最简单的统计模型来说已经相当不错！

| 概念 | 含义 |
|------|------|
| 贝叶斯定理 | 用先验概率和条件概率推断后验概率 |
| 朴素假设 | 词与词之间相互独立（简化计算）|
| Accuracy | 总体正确率 |
| Precision | 预测为某类时，真的是那类的比例 |
| Recall | 某类文章里，被正确找出的比例 |
| F1-Score | Precision 和 Recall 的调和平均 |
| 混淆矩阵 | 展示每种误判情况的数量 |

**下一章 →** Chapter 7：第二个分类器——kNN